<a href="https://colab.research.google.com/github/shaikmuzamil1844-coder/AIflow/blob/main/modified_Task1_Electron_Photon_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 GSoC 2026 — ML4SCI E2E | Common Task 1
## Electron vs Photon Classification using ResNet

**Author:** Shaik Muzamil  
**GitHub:** shaikmuzamil1844-coder  
**Organization:** ML4SCI  
**Project:** Sparse Neural Network Pipeline for Particle Collision Event Classification (E2E)

### Task Description
Build a model to classify **electron** vs **photon** events using 32×32 detector images with 2 channels:
- Channel 1: Hit Energy
- Channel 2: Hit Time

**Goal:** Maximize AUC score on the test set.

## 1. Setup & Imports

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
# Install dependencies when running in a fresh notebook runtime
import importlib.util
import subprocess
import sys

required = ['h5py', 'sklearn', 'matplotlib', 'seaborn']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f'Installing missing packages: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Dependencies already available.')


In [ ]:
# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


## 2. Download Dataset

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

DATASET_FILES = [
    'SinglePhotonPt50_IMGCROPS_n249k_RHv1.hdf5',
    'SingleElectronPt50_IMGCROPS_n249k_RHv1.hdf5',
]

def have_task1_data():
    return all(Path(name).exists() for name in DATASET_FILES)

def generate_synthetic_task1(path, label, samples=1500, image_size=32, channels=2):
    import h5py
    import numpy as np

    rng = np.random.default_rng(42 + label)
    images = np.zeros((samples, image_size, image_size, channels), dtype=np.float32)
    targets = np.full(samples, label, dtype=np.int64)
    center = image_size // 2

    for i in range(samples):
        spread = 3.2 if label == 1 else 5.5
        hits = rng.poisson(28 if label == 1 else 20)
        eta = np.clip(rng.normal(center, spread, hits).astype(int), 0, image_size - 1)
        phi = np.clip(rng.normal(center, spread, hits).astype(int), 0, image_size - 1)
        energy = rng.exponential(1.2 if label == 1 else 0.9, hits).astype(np.float32)
        time = rng.normal(0.4 if label == 1 else 0.7, 0.12, hits).astype(np.float32)
        np.add.at(images[i, :, :, 0], (eta, phi), energy)
        np.add.at(images[i, :, :, 1], (eta, phi), np.abs(time))

    with h5py.File(path, 'w') as f:
        f.create_dataset('X', data=images)
        f.create_dataset('y', data=targets)

if have_task1_data():
    print('Task 1 dataset already present.')
else:
    kaggle_user = os.getenv('KAGGLE_USERNAME')
    kaggle_key = os.getenv('KAGGLE_KEY')
    kaggle_ok = bool(kaggle_user and kaggle_key)
    kaggle_cli = shutil.which('kaggle')

    if kaggle_ok:
        if kaggle_cli is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
            kaggle_cli = shutil.which('kaggle')
        try:
            subprocess.check_call([
                kaggle_cli, 'datasets', 'download', '-d',
                'vishakkbhat/electron-vs-photons-ml4sci', '--force'
            ])
            import zipfile
            with zipfile.ZipFile('electron-vs-photons-ml4sci.zip') as zf:
                zf.extractall('.')
        except Exception as exc:
            print(f'Kaggle download failed: {exc}')

    if not have_task1_data():
        print('Dataset unavailable. Generating a small synthetic fallback dataset for notebook execution...')
        generate_synthetic_task1(DATASET_FILES[0], label=0)
        generate_synthetic_task1(DATASET_FILES[1], label=1)

for name in DATASET_FILES:
    path = Path(name)
    print(f'{name}: {"found" if path.exists() else "missing"}')


Dataset URL: https://www.kaggle.com/datasets/vishakkbhat/electron-vs-photons-ml4sci
License(s): unknown
100% 196M/196M [00:02<00:00, 75.0MB/s]

Archive:  electron-vs-photons-ml4sci.zip
  inflating: SingleElectronPt50_IMGCROPS_n249k_RHv1.hdf5  
  inflating: SinglePhotonPt50_IMGCROPS_n249k_RHv1.hdf5  
-rw-r--r-- 1 root root 123M Feb 28  2023 SingleElectronPt50_IMGCROPS_n249k_RHv1.hdf5
-rw-r--r-- 1 root root 115M Feb 28  2023 SinglePhotonPt50_IMGCROPS_n249k_RHv1.hdf5


In [ ]:
import h5py
import numpy as np

def load_hdf5_sample(path, max_samples=15000):
    with h5py.File(path, 'r') as f:
        print(f'Keys: {list(f.keys())}')
        X = f['X'][:max_samples]
        if 'y' in f:
            y = f['y'][:max_samples]
        else:
            y = np.zeros(len(X), dtype=np.int64)
    return X, y

# Files are expected in the current working directory.
X_photon, y_photon = load_hdf5_sample('SinglePhotonPt50_IMGCROPS_n249k_RHv1.hdf5')
X_electron, y_electron = load_hdf5_sample('SingleElectronPt50_IMGCROPS_n249k_RHv1.hdf5')

n_channels = min(X_photon.shape[-1], 2)
X_photon = X_photon[:, :, :, :n_channels]
X_electron = X_electron[:, :, :, :n_channels]

N = min(len(X_photon), len(X_electron))
X_photon = X_photon[:N]
X_electron = X_electron[:N]
labels_photon = np.zeros(N, dtype=np.int64)
labels_electron = np.ones(N, dtype=np.int64)

X_all = np.concatenate([X_photon, X_electron], axis=0)
y_all = np.concatenate([labels_photon, labels_electron], axis=0)

print(f'✅ X_all: {X_all.shape}, y_all: {y_all.shape}')
print(f'Class balance: {np.bincount(y_all)}')


Keys: ['X', 'y']
Keys: ['X', 'y']
✅ X_all: (30000, 32, 32, 2), y_all: (30000,)
Class balance: [15000 15000]


In [ ]:
# Verify variables exist (X_photon and X_electron should now be 15k samples each)
print(f'X_photon shape: {X_photon.shape}')
print(f'X_electron shape: {X_electron.shape}')

# Create combined dataset
# Label: 0 = photon, 1 = electron
# Ensure only first 2 channels are taken, and reshape if necessary

# Use first 2 channels: Energy + Time
# X_photon and X_electron already loaded with 2 channels by load_hdf5_sample
n_channels = min(X_photon.shape[-1], 2)
X_photon_processed = X_photon[:, :, :, :n_channels]
X_electron_processed = X_electron[:, :, :, :n_channels]

# Since load_hdf5_sample already limits to 15k, N should be 15000
N = X_photon_processed.shape[0]

labels_photon   = np.zeros(N, dtype=np.int64)   # 0
labels_electron = np.ones(N,  dtype=np.int64)   # 1

X_all = np.concatenate([X_photon_processed, X_electron_processed], axis=0)
y_all = np.concatenate([labels_photon, labels_electron], axis=0)

print(f'Combined dataset — X: {X_all.shape}, y: {y_all.shape}')
print(f'Class balance: {np.bincount(y_all)}')


X_photon shape: (15000, 32, 32, 2)
X_electron shape: (15000, 32, 32, 2)
Combined dataset — X: (30000, 32, 32, 2), y: (30000,)
Class balance: [15000 15000]


## 4. Data Preprocessing

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# Convert to (N, C, H, W) format for PyTorch
X_all_t = np.transpose(X_all, (0, 3, 1, 2)).astype(np.float32)

# Normalise each channel independently
for c in range(X_all_t.shape[1]):
    ch = X_all_t[:, c, :, :]
    mean, std = float(ch.mean()), float(ch.std()) + 1e-8
    X_all_t[:, c, :, :] = (ch - mean) / std
    print(f'Channel {c} (mean={mean:.4f}, std={std:.4f})')

# Train / Val / Test split 80 / 10 / 10
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all_t, y_all, test_size=0.20, random_state=42, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'\nTrain: {X_train.shape[0]:,}  |  Val: {X_val.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

def to_tensor(X, y):
    return TensorDataset(torch.tensor(X), torch.tensor(y))

train_ds = to_tensor(X_train, y_train)
val_ds = to_tensor(X_val, y_val)
test_ds = to_tensor(X_test, y_test)

cpu_workers = 0 if device.type == 'cpu' else 2
pin_memory = device.type == 'cuda'
batch_size = 128 if device.type == 'cpu' else 512

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=cpu_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=cpu_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=cpu_workers, pin_memory=pin_memory)

print('✅ DataLoaders ready!')


Channel 0 (mean=-0.0000, std=1.0000)
Channel 1 (mean=-0.0000, std=1.0000)

Train: 24,000  |  Val: 3,000  |  Test: 3,000
✅ DataLoaders ready!


## 5. Model — ResNet-15

In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block with BatchNorm and ReLU."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)


class ResNet15(nn.Module):
    """ResNet-15: lightweight ResNet matching Andrews et al. (2020)."""
    def __init__(self, in_channels=2, num_classes=1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.layer1 = self._make_layer(32,  64,  stride=1)
        self.layer2 = self._make_layer(64,  128, stride=2)
        self.layer3 = self._make_layer(128, 256, stride=2)
        self.layer4 = self._make_layer(256, 256, stride=2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.drop   = nn.Dropout(0.3)
        self.fc     = nn.Linear(256, num_classes)

    def _make_layer(self, in_c, out_c, stride):
        return nn.Sequential(
            ResidualBlock(in_c,  out_c, stride),
            ResidualBlock(out_c, out_c, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x).flatten(1)
        x = self.drop(x)
        return self.fc(x).squeeze(1)


model = ResNet15(in_channels=n_channels).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: ResNet-15')
print(f'Parameters: {total_params:,}')
print(model)

Model: ResNet-15
Parameters: 5,185,281
ResNet15(
  (stem): Sequential(
    (0): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (shortcut): Sequential(
        (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): ResidualBlock(
      (conv1): Conv2d(64, 64, ker

## 6. Training

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-6)

EPOCHS = 25 if device.type == 'cuda' else 3
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_auc = 0.0

def evaluate(loader):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.float().to(device)
            logits = model(X_batch)
            total_loss += criterion(logits, y_batch).item() * len(y_batch)
            all_logits.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())
    probs = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    auc = roc_auc_score(labels, probs)
    return total_loss / len(loader.dataset), auc, probs, labels

print(f'Training ResNet-15 for {EPOCHS} epochs...\n')
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Val Loss":>8} | {"Val AUC":>8}')
print('-' * 45)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.float().to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(y_batch)

    train_loss /= len(train_loader.dataset)
    val_loss, val_auc, _, _ = evaluate(val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'best_model_task1.pth')

    print(f'{epoch:>5} | {train_loss:>10.4f} | {val_loss:>8.4f} | {val_auc:>8.4f} {" ✅ best" if val_auc == best_auc else ""}')

print(f'\nBest Validation AUC: {best_auc:.4f}')


Training ResNet-15 for 25 epochs...

Epoch | Train Loss | Val Loss |  Val AUC
---------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


    1 |     0.6791 |   0.6589 |   0.6407  ✅ best


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


    2 |     0.6494 |   0.6440 |   0.6702  ✅ best


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 7. Evaluation on Test Set

In [ ]:
# Load best model and evaluate on test set
model.load_state_dict(torch.load('best_model_task1.pth', map_location=device))
test_loss, test_auc, test_probs, test_labels = evaluate(test_loader)
test_preds = (test_probs > 0.5).astype(int)
test_acc = accuracy_score(test_labels, test_preds)

print('=' * 40)
print('       TEST SET RESULTS')
print('=' * 40)
print(f'  AUC Score  : {test_auc:.4f}')
print(f'  Accuracy   : {test_acc:.4f}')
print(f'  Test Loss  : {test_loss:.4f}')
print('=' * 40)


In [ ]:
# ── Plots ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Training curves
ax = axes[0]
ax.plot(history['train_loss'], label='Train Loss', color='steelblue')
ax.plot(history['val_loss'],   label='Val Loss',   color='orange')
ax.set_title('Loss Curves', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.legend(); ax.grid(alpha=0.3)

# 2. AUC curve
ax = axes[1]
ax.plot(history['val_auc'], color='green', linewidth=2)
ax.axhline(test_auc, color='red', linestyle='--', label=f'Test AUC = {test_auc:.4f}')
ax.set_title('Validation AUC', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
ax.set_ylim([0.5, 1.0]); ax.legend(); ax.grid(alpha=0.3)

# 3. ROC Curve
ax = axes[2]
fpr, tpr, _ = roc_curve(test_labels, test_probs)
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {test_auc:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_title('ROC Curve — Test Set', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Task 1: Electron/Photon Classification — ResNet-15', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('task1_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Photon', 'Electron'],
            yticklabels=['Photon', 'Electron'], ax=ax)
ax.set_title(f'Confusion Matrix (Test AUC = {test_auc:.4f})', fontweight='bold')
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('task1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Final Test AUC: {test_auc:.4f}')

## 8. Summary

| Metric | Value |
|--------|-------|
| Model | ResNet-15 |
| In Channels | 2 (Energy + Time) |
| Image Size | 32×32 |
| Total Samples | 30,000 (15k each) |
| Train Samples | 24,000 |
| Val Samples | 3,000 |
| Test Samples | 3,000 |
| **Test AUC** | **(update after training)** |
| Optimizer | Adam (lr=5e-4) |
| Scheduler | CosineAnnealing |
| Epochs | 25 |
| Batch Size | 512 |